# 🥁 Drum OMR — MobileNetV3 Training

This notebook trains a model to read drum scores from bar images.

**What it learns:** Given a cropped image of one bar of drum notation,
output which drums are hit at each 32nd-note position, and the note duration.

**Architecture:** MobileNetV3-Small (pretrained on ImageNet) + two prediction heads

**Output:**
- Drum head: 32 beat positions × 14 drums = 448 binary values
- Duration head: 32 beat positions × 10 duration classes = 320 values

**Final JSON per bar** — an ordered list of notes, left-to-right:
```json
[
  {"duration": "eighth", "drums": ["kick", "hi_hat_closed"]},
  {"duration": "eighth", "drums": ["snare", "hi_hat_closed"]}
]
```
No beat positions in the output — just the sequence of notes in order.
Use the durations to reconstruct timing (e.g. two eighths fill one quarter-note beat).

In [ ]:
# ── 1. Install extra deps (torch/torchvision already on Colab) ────────────────
!pip install onnx onnxruntime onnxscript -q

In [ ]:
# ── 2. Mount Google Drive and unzip dataset ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

DRIVE_ROOT = '/content/drive/MyDrive/drumhub'
ZIP_PATH   = f'{DRIVE_ROOT}/dataset.zip'
DATA_ROOT  = '/content/dataset'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'

os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(DATA_ROOT):
    print('Unzipping dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print('Done.')
else:
    print('Dataset already extracted.')

IMG_DIR = f'{DATA_ROOT}/images'
LBL_DIR = f'{DATA_ROOT}/labels'
print(f'Images: {len(os.listdir(IMG_DIR))}  Labels: {len(os.listdir(LBL_DIR))}')

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
import torch

# BEAT_GRID: the 32 rhythmic positions we track in a bar, spaced one 32nd note apart.
# In 4/4 time: beat 1, 1.125, 1.25 ... 4.875
# Dividing each quarter-note beat into 8 equal slices captures 32nd notes correctly.
BEAT_GRID = [round(1 + i * 0.125, 3) for i in range(32)]

# DRUMS: every drum/cymbal the model can recognise.
# Order matters — it defines which output column maps to which drum.
# Grouped visually by frequency in the dataset so it is easy to see the balance.
DRUMS = [
    # ── High-frequency core drums (>1,000 hits in dataset) ──────────────────
    # These are the drums the model will learn most reliably.
    'hi_hat_closed',   # closed hi-hat: the tight "tss" sound
    'snare',           # snare drum: the "crack" on beats 2 and 4
    'kick',            # bass/kick drum: the low "thud" played with the foot
    'ride',            # ride cymbal: sustained "ping" pattern
    'crash',           # crash cymbal: accent hit
    'hi_hat_open_half', # half-open hi-hat: looser, washy sound
    'hi_hat_open_full', # fully open hi-hat: long sustain
    'hi_hat_pedal',    # hi-hat pedal (foot): closes the hi-hat with foot only
    'floor_tom_1',     # high floor tom (14"): deeper tone, sits on floor
    'floor_tom_2',     # low floor tom (16"): deepest tom, on floor
    'tom_mid',         # mid rack tom: middle of the kit
    'tom_hi',          # high rack tom: highest mounted tom
    'ride_bell',       # bell of the ride cymbal: bright "ding" sound
    'snare_rim',       # side stick / cross stick: quiet click on snare rim
    # 'snare_rimshot',   # rimshot: stick hits head and rim simultaneously, very loud crack
    # 'ride_tie',
    # 'splash',
    # 'sticks',
    # 'cowbell',
    # 'choked_crash',
    # 'china',
]
# removed: ride_tie, splash, sticks (<70 hits) + cowbell, clap, choked_crash, china
# (F1 = 0.000 after training — the model never learned to predict them, so they
#  only diluted the loss. Dropping them lets the 14 remaining drums learn better.)

# Build a lookup: drum name → column index (used in label_to_target below)
DRUM_IDX = {d: i for i, d in enumerate(DRUMS)}

# DURATIONS: the rhythmic length of each hit.
# The model predicts one duration class per beat position.
DURATIONS = [
    'whole', 'half', 'dotted_quarter', 'quarter',
    'dotted_eighth', 'eighth', 'sixteenth', 'thirty_second',
    'triplet_eighth', 'triplet_sixteenth',
]
DUR_IDX = {d: i for i, d in enumerate(DURATIONS)}

# Derived sizes — these flow through to the model automatically
N_BEATS     = len(BEAT_GRID)          # 32 positions per bar
N_DRUMS     = len(DRUMS)              # 14 drum types
N_DURATIONS = len(DURATIONS)          # 10 duration classes
N_DRUM_OUT  = N_BEATS * N_DRUMS       # 448 — one binary output per (beat, drum) pair  (14 drums × 32 beats)
N_DUR_OUT   = N_BEATS * N_DURATIONS   # 320 — one class output per beat position

IMG_H, IMG_W    = 128, 384  # image size fed to the model (height × width in pixels)
BATCH_SIZE      = 32        # number of bar images processed together each step
EPOCHS          = 15        # Phase 1 training epochs (backbone frozen)
FINETUNE_EPOCHS = 25        # Phase 2 training epochs (full model unfrozen)
LR              = 3e-4      # learning rate — how big each weight update step is
DUR_WEIGHT      = 0.5       # duration loss counts half as much as drum loss
THRESHOLD       = 0.5       # sigmoid score above this = "drum is present"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Drums: {N_DRUMS}  Drum output: {N_DRUM_OUT}  Duration output: {N_DUR_OUT}')


In [ ]:
# ── 4. Dataset class ──────────────────────────────────────────────────────────
import json
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T


def normalize_duration(dur_str: str) -> int:
    """Map a duration string to a DURATIONS index, with graceful fallback."""
    if dur_str in DUR_IDX:
        return DUR_IDX[dur_str]
    # Partial match for combos like 'dotted_triplet_eighth', 'sextuplet_eighth'
    for dur in DURATIONS:
        if dur in dur_str:
            return DUR_IDX[dur]
    return DUR_IDX['quarter']  # safe default


def label_to_target(label_path: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Convert a label JSON to two target tensors.

    y_drums : shape (N_DRUM_OUT,) = (448,)  float32
        A flat binary vector. Each position = 1 if that drum is hit at that beat.
        Example: if kick is hit at beat 1, position (beat_idx=0, drum_idx=2) = 1.0

    y_dur   : shape (N_BEATS,) = (32,)  int64
        One duration class index per beat position.
        Position = 0 ('whole') by default when no hit is present.
    """
    data    = json.loads(Path(label_path).read_text())
    y_drums = torch.zeros(N_BEATS, N_DRUMS)
    y_dur   = torch.zeros(N_BEATS, dtype=torch.long)

    for beat_entry in data.get('beats', []):
        # Rest beats ("rest": true) have no drums — skip them.
        # The model learns that all-zero at a position means silence.
        if beat_entry.get('rest'):
            continue

        # Find which 16th-note grid slot this beat falls into
        beat_val = float(beat_entry['beat'])
        dists    = [abs(beat_val - g) for g in BEAT_GRID]
        beat_idx = int(np.argmin(dists))

        # If the beat is more than half a 32nd-note step away from the nearest slot,
        # it is a triplet or unusual time signature beat — skip rather than mislabel
        if min(dists) > 0.07:
            continue

        y_dur[beat_idx] = normalize_duration(beat_entry.get('duration', 'quarter'))

        for drum in beat_entry.get('drums', []):
            # Ghost notes are labelled e.g. "snare_ghost" — a softer hit of the
            # same drum. Strip the suffix so they map to the same output column.
            base_drum = drum.replace('_ghost', '')
            if base_drum in DRUM_IDX:
                y_drums[beat_idx, DRUM_IDX[base_drum]] = 1.0
            # Unrecognised drums (e.g. leftover midi{n}) are silently ignored

    return y_drums.flatten(), y_dur   # (448,), (32,)


class DrumBarDataset(Dataset):
    """PyTorch Dataset that loads bar images and their drum labels.

    augment=True adds random small distortions during training so the model
    learns to handle slight variations in image quality and alignment.
    augment=False is used for validation and test — we want clean evaluation.
    """
    def __init__(self, image_paths: list, label_paths: list, augment: bool = False):
        self.images = image_paths
        self.labels = label_paths

        # Base transform: resize + convert to 3-channel + normalise
        # Normalise uses ImageNet mean/std because our backbone was pretrained on ImageNet
        base = [
            T.Resize((IMG_H, IMG_W)),
            T.Grayscale(num_output_channels=3),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
        # Augmentation transform: same as base but with extra noise
        aug = [
            T.Resize((IMG_H, IMG_W)),
            T.Grayscale(num_output_channels=3),
            T.RandomAffine(degrees=2, translate=(0, 0.04)),   # tiny rotation/shift
            T.ColorJitter(brightness=0.4, contrast=0.4),      # vary brightness
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),  # slight blur
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
        self.transform = T.Compose(aug if augment else base)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img            = Image.open(self.images[idx]).convert('L')
        x              = self.transform(img)
        y_drums, y_dur = label_to_target(self.labels[idx])
        return x, y_drums, y_dur

print('Dataset class defined.')


In [ ]:
# ── 5. Train / val / test split (by song, not by bar) ────────────────────────
import random
from collections import defaultdict

random.seed(42)

img_files = sorted(Path(IMG_DIR).glob('*.png'))
lbl_files = {p.stem: str(p) for p in Path(LBL_DIR).glob('*.json')}

songs = defaultdict(list)
for img in img_files:
    stem = img.stem
    song = stem[:stem.rfind('_bar')]
    if stem in lbl_files:
        songs[song].append((str(img), lbl_files[stem]))

song_names = list(songs.keys())
random.shuffle(song_names)

n_val  = max(1, int(len(song_names) * 0.10))
n_test = max(1, int(len(song_names) * 0.10))

test_songs  = set(song_names[:n_test])
val_songs   = set(song_names[n_test:n_test + n_val])
train_songs = set(song_names[n_test + n_val:])

def collect(song_set):
    imgs, lbls = [], []
    for s in song_set:
        for img, lbl in songs[s]:
            imgs.append(img)
            lbls.append(lbl)
    return imgs, lbls

train_imgs, train_lbls = collect(train_songs)
val_imgs,   val_lbls   = collect(val_songs)
test_imgs,  test_lbls  = collect(test_songs)

print(f'Songs  — train: {len(train_songs)}  val: {len(val_songs)}  test: {len(test_songs)}')
print(f'Bars   — train: {len(train_imgs)}   val: {len(val_imgs)}   test: {len(test_imgs)}')

In [ ]:
# ── 6. Data loaders ───────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

train_ds = DrumBarDataset(train_imgs, train_lbls, augment=True)
val_ds   = DrumBarDataset(val_imgs,   val_lbls,   augment=False)
test_ds  = DrumBarDataset(test_imgs,  test_lbls,  augment=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

x, y_drums, y_dur = next(iter(train_dl))
print(f'Batch — images: {x.shape}  drums: {y_drums.shape}  durations: {y_dur.shape}  active hits: {y_drums.sum().item():.0f}')

In [ ]:
# ── 7. Visualise one sample ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

img_raw        = Image.open(train_imgs[0]).convert('L')
y_drums, y_dur = label_to_target(train_lbls[0])
target         = y_drums.reshape(N_BEATS, N_DRUMS).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.imshow(img_raw, cmap='gray')
ax1.set_title(Path(train_imgs[0]).stem)
ax1.axis('off')

ax2.imshow(target.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks(range(N_BEATS))
ax2.set_xticklabels([str(b) for b in BEAT_GRID], rotation=45, fontsize=8)
ax2.set_yticks(range(N_DRUMS))
ax2.set_yticklabels(DRUMS, fontsize=8)
ax2.set_title('Target grid (blue = hit)')
ax2.set_xlabel('Beat position')

plt.tight_layout()
plt.show()

print('\nBeat   Duration            Drums')
print('-' * 55)
for bi, beat_val in enumerate(BEAT_GRID):
    if target[bi].sum() > 0:
        drums_hit = [DRUMS[di] for di in range(N_DRUMS) if target[bi, di] > 0]
        dur       = DURATIONS[y_dur[bi].item()]
        print(f'{beat_val:<7} {dur:<20} {drums_hit}')

In [ ]:
# ── 8. Model — two-head MobileNetV3 ──────────────────────────────────────────
#
# Architecture overview:
#   Input image (3 × 128 × 384)
#     → MobileNetV3-Small backbone  (pretrained on ImageNet — gives us free
#                                    edge/shape detection without training from scratch)
#     → Global average pool         (collapses spatial dimensions → single vector of 576 numbers)
#     → Shared head: Linear(576→1024) + Hardswish + Dropout
#          (Hardswish is an activation function — it adds non-linearity so the model
#           can learn complex patterns. Dropout randomly zeroes some neurons during
#           training to prevent overfitting — memorising rather than generalising.)
#     ├─→ drum_head:  Linear(1024 → 448)   sigmoid  → which drums are hit (32 beats × 14 drums)
#     └─→ dur_head:   Linear(1024 → 320)   softmax  → duration at each beat (32 beats × 10 classes)
#
# Note: the head sizes come from N_DRUM_OUT / N_DUR_OUT in cell 3, so they update
# automatically if you add or remove drums — no need to edit this cell.
#
# Two-phase training:
#   Phase 1: backbone frozen — only the two heads learn (fast, stable)
#   Phase 2: full model unfrozen — backbone fine-tunes on drum notation (slower, better)

import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class DrumOMRModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

        self.features = base.features              # CNN backbone — detects visual patterns
        self.avgpool  = base.avgpool               # global average pool
        self.shared   = nn.Sequential(             # shared Linear + activation + dropout
            *list(base.classifier[:-1])
        )
        feat_size = base.classifier[-1].in_features  # 1024
        self.drum_head = nn.Linear(feat_size, N_DRUM_OUT)   # → N_BEATS × N_DRUMS
        self.dur_head  = nn.Linear(feat_size, N_DUR_OUT)    # → N_BEATS × N_DURATIONS

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.shared(x)
        return self.drum_head(x), self.dur_head(x)


model = DrumOMRModel().to(DEVICE)

# Phase 1: freeze the backbone so only the heads train first.
# Reason: the pretrained backbone already knows how to detect edges and shapes.
# Training the heads first lets them stabilise before we touch the backbone.
for param in model.features.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameters — total: {total:,}  trainable (heads only): {trainable:,}  frozen (backbone): {total - trainable:,}')


In [ ]:
# ── 9. Training loop ──────────────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── pos_weight: handling class imbalance ──────────────────────────────────────
# Problem: kick is hit 55,000 times in the dataset; sticks only 15 times.
# If we treat all classes equally, the model learns to always predict sticks=0
# and still scores near-perfect on that class — but it has learned nothing useful.
#
# Solution: pos_weight tells the loss function "a missed sticks hit should be
# penalised much more heavily than a missed kick hit".
# Formula: pos_weight[i] = number_of_negatives[i] / number_of_positives[i]
# A drum present in 1% of beats gets a weight ~99x higher than one at 50%.
#
# This is built into PyTorch's BCEWithLogitsLoss via the pos_weight argument.

print('Computing pos_weight from training labels (this scans all training files)...')
all_drum_targets = torch.stack([label_to_target(l)[0] for l in train_lbls])

# Each column is one (beat position, drum) output with one observation per bar.
# Count positives/negatives per column, not across all N_BEATS positions again.
pos        = all_drum_targets.sum(0).clamp(min=1)   # clamp avoids division by zero
neg        = (len(train_lbls) - all_drum_targets.sum(0)).clamp(min=1)
# Clamp to max 50 — without this, rare drums like clap (502,688) and splash
# produce gradients so large the model cannot converge (val loss goes to 176,
# acc stays at 0.000 the entire training run).
# 50 still strongly upweights rare drums without breaking the loss landscape.
pos_weight = (neg / pos).clamp(max=50).to(DEVICE)

print(f'pos_weight range: {pos_weight.min():.1f} – {pos_weight.max():.1f}  (clamped to 50)')
print('(High weight = rare hit at this grid position. The loss penalises missing it more.)')
print('Weights at beat 1 (other positions have their own weights):')
print()
for i, drum in enumerate(DRUMS):
    print(f'  {drum:<20s}  pos_weight = {pos_weight[i]:.1f}')

# ── Loss functions ────────────────────────────────────────────────────────────
# BCEWithLogitsLoss: Binary Cross-Entropy for multi-label classification.
# "Multi-label" means each beat can have MULTIPLE drums at once (e.g. kick + hi-hat).
# This is different from multi-CLASS where only one answer is correct.
drum_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# CrossEntropyLoss: standard loss for single-class classification.
# Each beat has exactly ONE duration, so this is the right choice here.
dur_criterion  = nn.CrossEntropyLoss()

# ── Optimiser and scheduler ───────────────────────────────────────────────────
# AdamW: a popular gradient descent optimiser. It adjusts each weight individually
# based on how often it has been updated. weight_decay prevents weights from
# growing too large (another form of overfitting prevention).
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)

# CosineAnnealingLR: gradually reduces the learning rate over training following
# a cosine curve. Large steps early (explore), tiny steps late (fine-tune).
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(loader, train: bool, opt=None):
    """Run one full pass through the dataset (either training or evaluation).

    train=True:  model weights are updated (learning happens)
    train=False: weights are frozen, we just measure performance
    """
    if opt is None:
        opt = optimizer
    model.train(train)
    total_loss, drum_correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for x, y_drums, y_dur in loader:
            x, y_drums, y_dur = x.to(DEVICE), y_drums.to(DEVICE), y_dur.to(DEVICE)
            B = x.size(0)

            # Forward pass: feed images through the model, get predictions
            drum_logits, dur_logits = model(x)
            # logits = raw scores before sigmoid/softmax. Larger = more confident.

            # Drum loss: how wrong are we about which drums are hit?
            drum_loss = drum_criterion(drum_logits, y_drums)

            # Duration loss: only compute at beat positions that actually have hits.
            # There is no meaningful duration for a silent beat position.
            hit_mask = y_drums.view(B, N_BEATS, N_DRUMS).sum(dim=2) > 0  # (B, N_BEATS)
            if hit_mask.sum() > 0:
                dur_loss = dur_criterion(
                    dur_logits.view(B, N_BEATS, N_DURATIONS)[hit_mask],
                    y_dur[hit_mask],
                )
            else:
                dur_loss = torch.tensor(0.0, device=DEVICE)

            # Combined loss: drum loss + weighted duration loss
            loss = drum_loss + DUR_WEIGHT * dur_loss

            if train:
                opt.zero_grad()   # clear gradients from previous step
                loss.backward()   # compute gradients (backpropagation)
                opt.step()        # update weights

            total_loss  += loss.item() * B
            # Exact-bar accuracy: the entire bar is only "correct" if every
            # single drum prediction matches — a strict metric
            preds        = (drum_logits.sigmoid() > THRESHOLD).float()
            drum_correct += (preds == y_drums).all(dim=1).sum().item()
            total        += B

    return total_loss / total, drum_correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
CKPT_PATH = f'{CKPT_DIR}/omr_best.pt'

print('Starting Phase 1 training (backbone frozen — heads only)...')
print()

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    # Save the model whenever validation loss improves
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        torch.save(model.state_dict(), CKPT_PATH)
        saved = ' ← saved best checkpoint'
    else:
        saved = ''

    print(f'Epoch {epoch:2d}/{EPOCHS}  '
          f'train loss={tr_loss:.4f} acc={tr_acc:.3f}  '
          f'val loss={va_loss:.4f} acc={va_acc:.3f}{saved}')


In [ ]:
# ── 10. Plot Phase 1 training curves ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set_title('Phase 1 Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['train_acc'], label='train')
ax2.plot(history['val_acc'],   label='val')
ax2.set_title('Phase 1 Exact-bar accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 11. Evaluate Phase 1 on held-out test set ────────────────────────────────
from sklearn.metrics import f1_score

model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

all_drum_preds, all_drum_gts = [], []
all_dur_preds,  all_dur_gts  = [], []

with torch.no_grad():
    for x, y_drums, y_dur in test_dl:
        drum_logits, dur_logits = model(x.to(DEVICE))

        drum_preds = (drum_logits.sigmoid() > THRESHOLD).cpu().float()
        dur_preds  = dur_logits.view(-1, N_BEATS, N_DURATIONS).argmax(dim=2).cpu()

        all_drum_preds.append(drum_preds)
        all_drum_gts.append(y_drums)
        all_dur_preds.append(dur_preds)
        all_dur_gts.append(y_dur)

P      = torch.cat(all_drum_preds).numpy()
GT     = torch.cat(all_drum_gts).numpy()
P_dur  = torch.cat(all_dur_preds)
GT_dur = torch.cat(all_dur_gts)

exact_acc = (P == GT).all(axis=1).mean()
cell_acc  = (P == GT).mean()
print(f'Test exact-bar accuracy : {exact_acc:.3f} ({exact_acc*100:.1f}%)')
print(f'Test cell accuracy      : {cell_acc:.3f} ({cell_acc*100:.1f}%)')

# ── Sequence accuracy ─────────────────────────────────────────────────────────
# Exact-bar accuracy above requires every one of the 480 grid cells to match —
# it punishes a note that lands one beat-slot off as if the whole bar were wrong.
# But the decoder (cell 15) throws beat positions away and only keeps the ordered
# sequence of (drums, duration) events. Sequence accuracy measures THAT: does the
# left-to-right list of notes match? This is the metric that reflects what we ship.
def grid_to_sequence(drum_grid, dur_pred_row):
    """Convert one bar's (N_BEATS, N_DRUMS) grid into an ordered list of (drums, duration) tuples."""
    seq = []
    for bi in range(N_BEATS):
        # sort drums within a beat so {kick, hi_hat} == {hi_hat, kick} (order-independent)
        drums = tuple(sorted([DRUMS[di] for di in range(N_DRUMS) if drum_grid[bi, di] > 0.5]))
        if drums:
            seq.append((drums, int(dur_pred_row[bi])))
    return seq

P_grid  = P.reshape(-1, N_BEATS, N_DRUMS)
GT_grid = GT.reshape(-1, N_BEATS, N_DRUMS)

seq_matches = sum(
    grid_to_sequence(P_grid[i], P_dur[i].numpy()) == grid_to_sequence(GT_grid[i], GT_dur[i].numpy())
    for i in range(len(P))
)
seq_acc = seq_matches / len(P)
print(f'Test sequence accuracy  : {seq_acc:.3f} ({seq_acc*100:.1f}%)')
print()

P_drums  = P.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)
GT_drums = GT.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)

print(f'{"Drum":<18} {"F1":>6}')
print('-' * 26)
for i, drum in enumerate(DRUMS):
    f1 = f1_score(GT_drums[:, i], P_drums[:, i], zero_division=0)
    print(f'{drum:<18} {f1:6.3f}')

print()
hit_mask = GT.reshape(-1, N_BEATS, N_DRUMS).sum(axis=2) > 0
if hit_mask.sum() > 0:
    dur_acc = (P_dur.numpy()[hit_mask] == GT_dur.numpy()[hit_mask]).mean()
    print(f'Duration accuracy (at hit positions): {dur_acc:.3f} ({dur_acc*100:.1f}%)')
    print()
    gt_dur_flat = GT_dur.numpy()[hit_mask]
    print(f'{"Duration":<20} {"Count":>6}  Acc')
    print('-' * 36)
    for i, dur in enumerate(DURATIONS):
        count = (gt_dur_flat == i).sum()
        if count > 0:
            correct = ((P_dur.numpy()[hit_mask] == i) & (gt_dur_flat == i)).sum()
            print(f'{dur:<20} {count:>6}  {correct/count:.2f}')

In [ ]:
# ── 12. Visualise predictions ─────────────────────────────────────────────────
n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(15, n_show * 3))

model.eval()
with torch.no_grad():
    for i in range(n_show):
        img_raw = Image.open(test_imgs[i]).convert('L')
        x, y_drums, y_dur = test_ds[i]
        drum_logits, dur_logits = model(x.unsqueeze(0).to(DEVICE))

        pred  = (drum_logits.sigmoid() > THRESHOLD).cpu().float().reshape(N_BEATS, N_DRUMS)
        truth = y_drums.reshape(N_BEATS, N_DRUMS)

        axes[i, 0].imshow(img_raw, cmap='gray'); axes[i, 0].axis('off')
        axes[i, 0].set_title(Path(test_imgs[i]).stem[:40], fontsize=8)

        axes[i, 1].imshow(truth.numpy().T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
        axes[i, 1].set_title('Ground truth')
        axes[i, 1].set_yticks(range(N_DRUMS)); axes[i, 1].set_yticklabels(DRUMS, fontsize=7)

        axes[i, 2].imshow(pred.numpy().T, aspect='auto', cmap='Oranges', vmin=0, vmax=1)
        axes[i, 2].set_title('Predicted')
        axes[i, 2].set_yticks(range(N_DRUMS)); axes[i, 2].set_yticklabels(DRUMS, fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── 13. Unfreeze backbone and fine-tune everything (Phase 2) ──────────────────
# Start from best Phase 1 checkpoint, unfreeze all layers, train at LR/10.
# This lets the backbone adapt its features specifically to drum notation.

model.load_state_dict(torch.load(CKPT_PATH))

for param in model.parameters():
    param.requires_grad = True

trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All {trainable2:,} parameters now trainable')

CKPT_PATH_FT = f'{CKPT_DIR}/omr_finetuned.pt'
optimizer2   = AdamW(model.parameters(), lr=LR / 10, weight_decay=1e-4)
scheduler2   = CosineAnnealingLR(optimizer2, T_max=FINETUNE_EPOCHS)

history2 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_ft_loss = float('inf')

for epoch in range(1, FINETUNE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True,  opt=optimizer2)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    scheduler2.step()

    history2['train_loss'].append(tr_loss)
    history2['val_loss'].append(va_loss)
    history2['train_acc'].append(tr_acc)
    history2['val_acc'].append(va_acc)

    if va_loss < best_ft_loss:
        best_ft_loss = va_loss
        torch.save(model.state_dict(), CKPT_PATH_FT)
        saved = ' ← saved'
    else:
        saved = ''

    print(f'Finetune {epoch:2d}/{FINETUNE_EPOCHS}  '
          f'train loss={tr_loss:.4f} acc={tr_acc:.3f}  '
          f'val loss={va_loss:.4f} acc={va_acc:.3f}{saved}')

CKPT_PATH = CKPT_PATH_FT
print(f'\nBest finetuned model: {CKPT_PATH}')

In [ ]:
# ── 13a. Plot Phase 2 training curves ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history2['train_loss'], label='train')
ax1.plot(history2['val_loss'],   label='val')
ax1.set_title('Phase 2 Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history2['train_acc'], label='train')
ax2.plot(history2['val_acc'],   label='val')
ax2.set_title('Phase 2 Exact-bar accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 13b. Re-evaluate after Phase 2 ───────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

all_drum_preds, all_drum_gts = [], []
all_dur_preds,  all_dur_gts  = [], []

with torch.no_grad():
    for x, y_drums, y_dur in test_dl:
        drum_logits, dur_logits = model(x.to(DEVICE))

        drum_preds = (drum_logits.sigmoid() > THRESHOLD).cpu().float()
        dur_preds  = dur_logits.view(-1, N_BEATS, N_DURATIONS).argmax(dim=2).cpu()

        all_drum_preds.append(drum_preds)
        all_drum_gts.append(y_drums)
        all_dur_preds.append(dur_preds)
        all_dur_gts.append(y_dur)

P      = torch.cat(all_drum_preds).numpy()
GT     = torch.cat(all_drum_gts).numpy()
P_dur  = torch.cat(all_dur_preds)
GT_dur = torch.cat(all_dur_gts)

exact_acc = (P == GT).all(axis=1).mean()
cell_acc  = (P == GT).mean()
print(f'Test exact-bar accuracy : {exact_acc:.3f} ({exact_acc*100:.1f}%)')
print(f'Test cell accuracy      : {cell_acc:.3f} ({cell_acc*100:.1f}%)')

# ── Sequence accuracy ─────────────────────────────────────────────────────────
# The metric that reflects what we actually ship: does the ordered left-to-right
# list of (drums, duration) events match? Beat-slot position is ignored.
def grid_to_sequence(drum_grid, dur_pred_row):
    """Convert one bar's (N_BEATS, N_DRUMS) grid into an ordered list of (drums, duration) tuples."""
    seq = []
    for bi in range(N_BEATS):
        drums = tuple(sorted([DRUMS[di] for di in range(N_DRUMS) if drum_grid[bi, di] > 0.5]))
        if drums:
            seq.append((drums, int(dur_pred_row[bi])))
    return seq

P_grid  = P.reshape(-1, N_BEATS, N_DRUMS)
GT_grid = GT.reshape(-1, N_BEATS, N_DRUMS)

seq_matches = sum(
    grid_to_sequence(P_grid[i], P_dur[i].numpy()) == grid_to_sequence(GT_grid[i], GT_dur[i].numpy())
    for i in range(len(P))
)
seq_acc = seq_matches / len(P)
print(f'Test sequence accuracy  : {seq_acc:.3f} ({seq_acc*100:.1f}%)')
print()

P_drums  = P.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)
GT_drums = GT.reshape(-1, N_BEATS, N_DRUMS).max(axis=1)

print(f'{"Drum":<18} {"F1":>6}')
print('-' * 26)
for i, drum in enumerate(DRUMS):
    f1 = f1_score(GT_drums[:, i], P_drums[:, i], zero_division=0)
    print(f'{drum:<18} {f1:6.3f}')

print()
hit_mask = GT.reshape(-1, N_BEATS, N_DRUMS).sum(axis=2) > 0
if hit_mask.sum() > 0:
    dur_acc = (P_dur.numpy()[hit_mask] == GT_dur.numpy()[hit_mask]).mean()
    print(f'Duration accuracy (at hit positions): {dur_acc:.3f} ({dur_acc*100:.1f}%)')
    print()
    gt_dur_flat = GT_dur.numpy()[hit_mask]
    print(f'{"Duration":<20} {"Count":>6}  Acc')
    print('-' * 36)
    for i, dur in enumerate(DURATIONS):
        count = (gt_dur_flat == i).sum()
        if count > 0:
            correct = ((P_dur.numpy()[hit_mask] == i) & (gt_dur_flat == i)).sum()
            print(f'{dur:<20} {count:>6}  {correct/count:.2f}')

In [ ]:
# ── 13c. Save evaluation results to disk ──────────────────────────────────────
# Cell 13b prints its metrics and nothing else, so the numbers vanish when the
# runtime is recycled and the model card has to quote figures nobody can check.
# This cell writes the same metrics to a JSON artifact next to the checkpoint.
#
# It recomputes nothing — it serialises the arrays 13b left in memory, so run
# 13b first. Committing the resulting file is what makes a claimed number
# reproducible.

import datetime
import subprocess

EVAL_PATH = f'{CKPT_DIR}/eval_results.json'

# Per-drum F1, bar level — identical definition to 13b
per_drum_f1 = {
    drum: float(f1_score(GT_drums[:, i], P_drums[:, i], zero_division=0))
    for i, drum in enumerate(DRUMS)
}

# Duration accuracy at hit positions only, with the support behind each class.
# Support matters: a 100% duration read on 9 examples is noise, not a result.
gt_dur_flat = GT_dur.numpy()[hit_mask]
p_dur_flat  = P_dur.numpy()[hit_mask]
per_duration = {}
for i, dur in enumerate(DURATIONS):
    count = int((gt_dur_flat == i).sum())
    if count > 0:
        correct = int(((p_dur_flat == i) & (gt_dur_flat == i)).sum())
        per_duration[dur] = {
            'support':  count,
            'correct':  correct,
            'accuracy': correct / count,
        }

try:  # records which code produced these numbers; harmless if git is absent
    commit = subprocess.check_output(
        ['git', 'rev-parse', '--short', 'HEAD'], stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    commit = None

results = {
    'generated_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'git_commit':   commit,
    'checkpoint':   CKPT_PATH,
    'config': {
        'drums':       DRUMS,
        'durations':   DURATIONS,
        'n_beats':     N_BEATS,
        'n_drums':     N_DRUMS,
        'n_durations': N_DURATIONS,
        'threshold':   THRESHOLD,
        'img_h':       IMG_H,
        'img_w':       IMG_W,
    },
    'dataset': {
        'test_bars':    int(len(P)),
        'test_songs':   len(test_songs) if 'test_songs' in dir() else None,
        'split':        'by song, not by bar',
    },
    'metrics': {
        # Ordered weakest-to-strongest claim, so the honest one reads first.
        'sequence_accuracy':  float(seq_acc),   # what the product actually ships
        'exact_bar_accuracy': float(exact_acc), # every slot in the grid correct
        'cell_accuracy':      float(cell_acc),  # per-slot; empty slots dominate it
        'duration_accuracy_at_hits': float(dur_acc),
        'per_drum_f1':        per_drum_f1,
        'per_duration':       per_duration,
    },
}

with open(EVAL_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved to {EVAL_PATH}\n')
print(f'  sequence accuracy   {seq_acc:.3f}   <- lead with this one')
print(f'  exact-bar accuracy  {exact_acc:.3f}')
print(f'  cell accuracy       {cell_acc:.3f}')
print(f'  duration @ hits     {dur_acc:.3f}')
print(f'  test bars           {len(P)}')

# Markdown for the model card's Performance section — paste it straight in.
print('\n--- paste into ml/README.md ---\n')
print(f'| Metric | Value |')
print(f'| --- | --- |')
print(f'| Sequence accuracy | {seq_acc*100:.1f}% |')
print(f'| Exact-bar accuracy | {exact_acc*100:.1f}% |')
print(f'| Cell accuracy | {cell_acc*100:.1f}% |')
print(f'| Duration accuracy (at hits) | {dur_acc*100:.1f}% |')
print(f'\nEvaluated on {len(P)} bars from songs held out of training.\n')
print(f'| Drum | F1 |')
print(f'| --- | --- |')
for drum, f1 in sorted(per_drum_f1.items(), key=lambda kv: -kv[1]):
    print(f'| {drum} | {f1:.3f} |')


In [ ]:
# ── 14. Export to ONNX ────────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

ONNX_PATH = f'{CKPT_DIR}/omr.onnx'

model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

dummy_input = torch.randn(1, 3, IMG_H, IMG_W).to(DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=['image'],
    output_names=['drum_logits', 'dur_logits'],
    dynamic_axes={
        'image':       {0: 'batch'},
        'drum_logits': {0: 'batch'},
        'dur_logits':  {0: 'batch'},
    },
    opset_version=18,
)

# Write a sidecar config so consumers never need to hardcode any model dimensions.
# Everything the inference code needs — image size, drum names, thresholds — lives here.
CONFIG_PATH = f'{CKPT_DIR}/omr_config.json'
config = {
    "DRUMS":       DRUMS,
    "DURATIONS":   DURATIONS,
    "N_BEATS":     N_BEATS,
    "N_DRUMS":     N_DRUMS,
    "N_DURATIONS": N_DURATIONS,
    "IMG_H":       IMG_H,
    "IMG_W":       IMG_W,
    "THRESHOLD":   THRESHOLD,
}
json.dump(config, open(CONFIG_PATH, "w"), indent=2)
print(f'Config saved to: {CONFIG_PATH}')

sess = ort.InferenceSession(ONNX_PATH)
dummy_np = dummy_input.cpu().numpy()
ort_drum, ort_dur = sess.run(None, {'image': dummy_np})
with torch.no_grad():
    pt_drum, pt_dur = model(dummy_input)

drum_diff = abs(ort_drum - pt_drum.cpu().numpy()).max()
dur_diff  = abs(ort_dur  - pt_dur.cpu().numpy()).max()
print(f'ONNX exported to: {ONNX_PATH}')
print(f'Max diff — drums: {drum_diff:.2e}  duration: {dur_diff:.2e}')


In [ ]:
# ── 14a. Verify the ONNX export ───────────────────────────────────────────────
# The .onnx that was in the repo was 284 KB against ~9.2 MB expected, so it was
# stale or a partial download and nobody noticed. This cell makes that failure
# loud: it checks the file on disk is the size the parameter count implies, and
# raises if it is not.
#
# Run it straight after cell 14. It is cheap and it is the difference between
# shipping a broken artifact and knowing you did.

import os

param_count = sum(p.numel() for p in model.parameters())
size_bytes  = os.path.getsize(ONNX_PATH)
size_mb     = size_bytes / 1024**2
expected_mb = param_count * 4 / 1024**2   # fp32 = 4 bytes per parameter

print(f'Parameters      : {param_count:,}')
print(f'Expected (fp32) : ~{expected_mb:.2f} MB')
print(f'Actual on disk  : {size_mb:.2f} MB  ({size_bytes:,} bytes)')

# ONNX carries graph overhead on top of the weights, so allow a generous band.
# The failure being caught here is order-of-magnitude, not a few percent.
lo, hi = expected_mb * 0.75, expected_mb * 1.5 + 5
if not (lo <= size_mb <= hi):
    raise RuntimeError(
        f'ONNX export looks wrong: {size_mb:.2f} MB is outside the expected '
        f'{lo:.2f}-{hi:.2f} MB band for {param_count:,} fp32 parameters. '
        f'Re-export before using this file, and do not quote a model size from it.'
    )
print('\nSize check: PASS')

# Re-confirm the parity check from cell 14 on a fresh session, so the file that
# is actually on disk is the one being validated — not the in-memory export.
sess_check = ort.InferenceSession(ONNX_PATH)
probe = torch.randn(1, 3, IMG_H, IMG_W)
ort_d, ort_u = sess_check.run(None, {'image': probe.numpy()})
model.eval()
with torch.no_grad():
    pt_d, pt_u = model(probe.to(DEVICE))

drum_diff = float(abs(ort_d - pt_d.cpu().numpy()).max())
dur_diff  = float(abs(ort_u - pt_u.cpu().numpy()).max())
print(f'Parity vs PyTorch — drums: {drum_diff:.2e}  duration: {dur_diff:.2e}')

TOL = 1e-4
if max(drum_diff, dur_diff) > TOL:
    raise RuntimeError(f'ONNX output drifted from PyTorch beyond {TOL:.0e}.')
print('Parity check: PASS')

print(f'\nDownload these three together — the config is the inference contract:')
print(f'  {ONNX_PATH}')
print(f'  {CKPT_PATH}')
print(f'  {CKPT_DIR}/omr_config.json')


In [ ]:
# ── 15. Decode ONNX output back to JSON ───────────────────────────────────────
# This is the function you will call in Electron / FastAPI:
#   image → ONNX inference → ordered list of {duration, drums}
#
# No beat positions in the output — just the sequence of notes left-to-right.
# The durations tell you how long each note lasts; that is all you need to
# reconstruct the rhythm and feed notes into a drum machine or score editor.

import torchvision.transforms as transforms

def predict_bar(image_path: str) -> list:
    """Run ONNX inference on one bar image.

    Returns an ordered list of notes:
        [{duration, drums}, {duration, drums}, ...]

    Each entry = one rhythmic position that has at least one drum hit.
    Entries are ordered left-to-right as they appear in the bar.
    Silent positions (rests) are omitted — the durations fill the bar time.
    """
    transform = transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    img  = Image.open(image_path).convert('L')
    x    = transform(img).unsqueeze(0).numpy()          # (1, 3, H, W)

    drum_logits, dur_logits = sess.run(None, {'image': x})

    # sigmoid converts raw logits → 0–1 probability that each drum is present
    drum_probs = 1 / (1 + np.exp(-drum_logits[0]))
    drum_grid  = drum_probs.reshape(N_BEATS, N_DRUMS)   # (32, 14)

    dur_grid  = dur_logits[0].reshape(N_BEATS, N_DURATIONS)
    dur_preds = dur_grid.argmax(axis=1)                 # pick the top duration per beat

    notes = []
    for bi in range(N_BEATS):
        drums = [DRUMS[di] for di in range(N_DRUMS) if drum_grid[bi, di] > THRESHOLD]
        if drums:
            notes.append({
                'duration': DURATIONS[dur_preds[bi]],
                'drums':    drums,
            })

    return notes


# Test on a few samples to verify output format
for idx in range(min(3, len(test_imgs))):
    notes = predict_bar(test_imgs[idx])
    if notes:
        print(f'--- {Path(test_imgs[idx]).stem} ---')
        print(json.dumps(notes, indent=2))
        break

## Next steps

1. Download `omr.onnx` from Drive to your Mac
2. Load it in FastAPI: `onnxruntime.InferenceSession('omr.onnx')`
3. Electron sends a bar image → FastAPI calls `predict_bar()` → returns ordered note list
4. Each note is `{duration, drums}` — feed directly into your drum machine / VexFlow
5. To improve weak drums (floor_tom_2, hi_hat_pedal): add more training songs and retrain
